## Objetivo da Análise

Esta análise busca responder:
“Quais fatores influenciam o preço e a demanda dos anúncios do Airbnb no Rio de Janeiro?”

In [ ]:
import pandas as pd

df = pd.read_csv("listings.csv")

df.head()

In [ ]:
cidade = "Rio de Janeiro"
print(f"Cidade analisada: {cidade}")

In [ ]:
#colunas, linhas
df.shape

In [ ]:

#colunas
df.columns
#Tipo de dado de cada coluna
df.dtypes

In [ ]:
#Ranking dos que possuem mais valores nulos
df.isnull().sum().sort_values(ascending=False)

In [ ]:
colunas_relevantes = [
    'price',
    'room_type', 
    'neighbourhood_cleansed',
    'accommodates',
    'number_of_reviews', 
    'reviews_per_month',  
    'amenities'                   
]

In [ ]:
df[colunas_relevantes].info()

### Tratamento de PRICE
**Justificativa de Price:** Realizei o tratamento dos dados na coluna PRICE, que possuia caracteres especiais. Os removi e converti a coluna em float para melhorar a ánalise com base nos preços

In [ ]:
df['price'].head(10)

In [ ]:
df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)

In [ ]:
df['price'].head(10)

In [ ]:
df[colunas_relevantes].head(5)

In [ ]:
df['price'].isnull().sum()

In [ ]:
df = df.dropna(subset=['price']).copy()

In [ ]:
df['price'].isnull().sum()

### Transformação de amenities
**Justificativa das amenities:** Optei por extrair as 5 amenities mais frequentes na base de dados pois elas representam a infraestrutura básica e o padrão mínimo esperado pela maioria dos hóspedes

In [ ]:
# 1. Removemos os colchetes e aspas da string usando regex
amenities_limpas = df['amenities'].str.replace(r'[\[\]"]', '', regex=True)

# 2. Separamos o texto por vírgulas para criar uma lista
amenities_listas = amenities_limpas.str.split(', ')

# 3.separa cada item da lista em uma linha temporária.
ranking_amenities = amenities_listas.explode().value_counts()

# Mostrando amenities que mais aparecem no Rio de Janeiro
display(ranking_amenities.head(5))

In [ ]:
top_amenities = ranking_amenities.head(5).index.tolist()

import re
colunas_novas = []

for amenidade in top_amenities:
    nome_limpo = re.sub(r'[^a-zA-Z0-9]', '_', amenidade.lower())
    nome_coluna = f'has_{nome_limpo}'
    
    df[nome_coluna] = df['amenities'].str.contains(amenidade, case=False, na=False).astype(int)
    
    colunas_novas.append(nome_coluna)

df[['price', 'accommodates'] + colunas_novas].head(10)    

In [ ]:
resumo = df.groupby('room_type').agg({
    'price': ['mean', 'median'],
    'number_of_reviews': 'mean',
    'accommodates': 'mean'
})
print(resumo)

In [ ]:
df.groupby('neighbourhood_cleansed')['price'].mean().sort_values(ascending=False).head(5)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig1, axes1 = plt.subplots(1, 2, figsize=(14, 5))

# 1. Distribuição de preços
df_sem_outlier = df[df['price'] < df['price'].quantile(0.95)]
sns.histplot(df_sem_outlier['price'], bins=60, ax=axes1[0], color='steelblue')
axes1[0].set_title('Distribuição de Preços')
axes1[0].set_xlabel('Preço (R$)')

# 2. Preço médio por tipo de quarto
medias = df.groupby('room_type')['price'].median().sort_values()
medias.plot(kind='barh', ax=axes1[1], color='coral')
axes1[1].set_title('Preço Mediano por Tipo de Quarto')
axes1[1].set_xlabel('Preço Mediano (R$)')

plt.tight_layout()
plt.show() 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Média
top_mean = df.groupby('neighbourhood_cleansed')['price'].mean().nlargest(5)
top_mean.plot(kind='bar', ax=axes[0], color='orange')
axes[0].set_title('Top 5 — Média (com outliers)')
axes[0].tick_params(axis='x', rotation=45)

# Mediana
top_median = df.groupby('neighbourhood_cleansed')['price'].median().nlargest(5)
top_median.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Top 5 — Mediana')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
df_estacio = df[df['neighbourhood_cleansed'] == 'Estácio']

media_estacio = df_estacio['price'].mean()
mediana_estacio = df_estacio['price'].median()

moda_estacio = df_estacio['price'].mode()[0] 

print(f"--- Valores de Preço no Bairro Estácio ---")
print(f"Média (Average): R$ {media_estacio:.2f}")
print(f"Mediana (Valor do meio): R$ {mediana_estacio:.2f}")
print(f"Moda (Preço mais comum): R$ {moda_estacio:.2f}")

anuncios_caros = df_estacio.sort_values(by='price', ascending=False)

display(anuncios_caros[['id', 'name', 'room_type', 'price']].head(5))

In [ ]:
#Proxy de demanda = number_of_reviews
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x='price', y='number_of_reviews', ax=axes[0])
axes[0].set_xlim(0, 5000)
axes[0].set_title('Preço vs Demanda (até R$5000)')

sns.scatterplot(data=df, x='price', y='number_of_reviews', ax=axes[1])
axes[1].set_xlim(0, 1000)
axes[1].set_title('Preço vs Demanda (até R$1000)')

plt.tight_layout()
plt.show()

### Gráfico 1 - Distribuição de preços: 
A distribuição de preços é assimétrica à direita, com a maioria dos anúncios concentrada entre R$100 e R$500. O pico está  em torno de R$200-R$300, indicando que esse é o preço mais comum no mercado. A cauda longa à direita mostra a existência de imóveis de luxo que elevam a média, justificando o uso da mediana como medida mais representativa. (Caso utilize a média, a conta seria altamente influneciada)

### Gráfico 2 — Preço Mediano por Tipo de Quarto:
Hotel room apresenta o maior preço mediano (~R$450), seguido de Entire home/apt (R$350). Private room (R$200) e Shared room (R$100) são significativamente mais baratos. Isso reflete a lógica do mercado: acomodações completas e com serviços agregados comandam preços maiores, enquanto quartos compartilhados atendem um público mais sensível ao preço.

### Gráfico 3 — Top 5 Bairros por Preço Médio:
A média indica Estácio e Joá como os bairros mais caros, porém há distorção nos dados. No caso do Estácio, um valor extremamente alto (outlier) eleva artificialmente a média, não representando a realidade do bairro.

### Gráfico 3.1 — Top 5 Bairros por Preço mediano:
A mediana elimina o impacto de outliers e mostra valores mais realistas. Assim, os bairros apresentados refletem melhor o preço típico das diárias.

### Gráfico 4 - Demanda de procura
A quantidade de demanda está diretamente relacionada ao preço das diárias, a busca por diárias em valores mais acessíveis (R$150 -R$ 450) é muito maior do que para valores mais elevados

In [ ]:
# --- Teste: amenities vs Preço ---

df['qtd_amenities'] = df[colunas_novas].sum(axis=1)
baixa = df[df['qtd_amenities'] <= 2]['price']
alta = df[df['qtd_amenities'] > 2]['price']

from scipy import stats

# H0: preço é igual independentemente da quantidade de amenities
# H1: imóveis com mais amenities são mais caros

stat, p = stats.mannwhitneyu(alta, baixa, alternative='greater')

print("--- Teste: amenities vs Preço ---")
print("amenities consideradas:", colunas_novas)
print(f"p-valor: {p:.4f}")

if p < 0.05:
    print("Conclusão: Imóveis com mais amenities são significativamente mais caros.")
else:
    print("Conclusão: Não há diferença estatística no preço.")

In [ ]:
# --- Teste 2: Bairros têm preços diferentes? ---
# H0: preço médio é igual entre os bairros
# H1: pelo menos um bairro tem preço diferente

# Agrupando apenas bairros com um volume bom de anúncios (mais de 30)
grupos_bairros = [g['price'].values for _, g in df.groupby('neighbourhood_cleansed') if len(g) >= 30]

# Calculando a estatística
stat, p = stats.kruskal(*grupos_bairros)

print(f"\n--- Teste 2: Impacto da Localização no Preço ---")
print(f"p-valor: {p:.4f}")

if p < 0.05:
    print("Conclusão: O bairro afeta fortemente o valor da diária!")
    print("Existe uma diferença de preços real e estatisticamente comprovada entre as diferentes regiões.")
else:
    print("Conclusão: NÃO existe diferença de preço comprovada entre os bairros.")
    print("Estatisticamente, os anfitriões cobram valores parecidos independente da região analisada.")

In [ ]:
from scipy import stats

# --- Teste 3: Superhost vs Demanda ---
# H0: número de reviews é igual entre superhosts e comuns
# H1: superhosts têm mais reviews (maior demanda)

superhosts = df[df['host_is_superhost'] == 't']['number_of_reviews']
comuns = df[df['host_is_superhost'] == 'f']['number_of_reviews']

stat, p = stats.mannwhitneyu(superhosts, comuns, alternative='greater')

print("--- Teste 3: Superhost vs Demanda ---")
print(f"p-valor: {p:.4f}")

if p < 0.05:
    print("Conclusão: Superhosts têm significativamente mais avaliações.")
    print("Isso indica maior demanda para esses anúncios.")
else:
    print("Conclusão: Não há diferença estatisticamente significativa.")

In [ ]:
# --- Diferença entre superhost e comuns ---
agg_dict = {
    'id': 'count',
    'price': 'median',
    'number_of_reviews': 'mean'
}
for col in colunas_novas:
    agg_dict[col] = 'mean'
resumo_host = df.groupby('host_is_superhost').agg(agg_dict)
resumo_host = resumo_host.rename(columns={
    'id': 'total_anuncios',
    'price': 'preco_mediano',
    'number_of_reviews': 'reviews_medio'
})
for col in colunas_novas:
    resumo_host = resumo_host.rename(columns={col: f'perc_{col.replace("has_", "")}'})
colunas_perc = [col for col in resumo_host.columns if col.startswith('perc_')]
resumo_host[colunas_perc] = (resumo_host[colunas_perc] * 100).round(1)
resumo_host['preco_mediano'] = resumo_host['preco_mediano'].round(2)
resumo_host['reviews_medio'] = resumo_host['reviews_medio'].round(1)

display(resumo_host)

## Conclusão

A análise dos dados do Airbnb no Rio de Janeiro indica que o preço das diárias é influenciado principalmente pelo tipo de acomodação, localização e quantidade de amenities.Bairros mais valorizados apresentam preços mais elevados.

Em relação à demanda, observou-se que anúncios com preços mais acessíveis concentram maior número de avaliações, sugerindo maior procura. Além disso, anfitriões superhost tendem a apresentar maior demanda, indicando maior atratividade ou qualidade percebida.

Os testes estatísticos confirmaram que:
- A quantidade de amenities influencia o preço dos anúncios;
- O bairro impacta significativamente o valor das diárias;
- Superhosts possuem maior número de avaliações, indicando maior demanda.

Como limitação, destaca-se a presença de outliers nos preços e o uso do número de avaliações como proxy de demanda, que não representa perfeitamente o número real de reservas.